# AI Termika — Baseline (data-only MLP)

Jednoduchá MLP síť trénovaná pouze na měřených datech, bez fyzikální loss.

Tento baseline ukazuje výchozí bod — studenti by měli přidat physics loss (PINN) pro lepší výsledky.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používám: {device}")

torch.manual_seed(42)
np.random.seed(42)

## 1. Načtení dat

In [ ]:
DATA_DIR = "ai_termika_dataset/student"

train_df = pd.read_csv(f"{DATA_DIR}/train_measurements.csv")
boundary_df = pd.read_csv(f"{DATA_DIR}/boundary_partial.csv")
initial_df = pd.read_csv(f"{DATA_DIR}/initial_sparse.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test_points.csv")

with open(f"{DATA_DIR}/constants.json") as f:
    constants = json.load(f)

print(f"Train bodů: {len(train_df)}, Boundary bodů: {len(boundary_df)}, Initial bodů: {len(initial_df)}")
print(f"Test bodů: {len(test_df)}")

# Spojíme všechna dostupná data do jednoho datasetu
all_data = pd.concat([train_df, boundary_df, initial_df], ignore_index=True)
print(f"Celkem trénovacích bodů: {len(all_data)}")

In [ ]:
def df_to_tensors(df, cols_x, col_y=None):
    X = torch.tensor(df[cols_x].values, dtype=torch.float32, device=device)
    if col_y:
        y = torch.tensor(df[col_y].values, dtype=torch.float32, device=device).unsqueeze(1)
        return X, y
    return X

X_all, y_all = df_to_tensors(all_data, ["x", "t"], "u")
X_test = df_to_tensors(test_df, ["x", "t"])

## 2. Vizualizace dat

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(train_df["t"], train_df["x"], c=train_df["u"], cmap="inferno", s=20, label="train")
ax.scatter(boundary_df["t"], boundary_df["x"], c="red", marker="s", s=30, label="boundary")
ax.scatter(initial_df["t"], initial_df["x"], c="green", marker="^", s=30, label="initial")
ax.set_xlabel("t")
ax.set_ylabel("x")
ax.set_title("Rozmístění datových bodů")
ax.legend()

ax = axes[1]
ax.scatter(initial_df["x"], initial_df["u"], c="green", marker="^", s=50, label="initial")
ax.set_xlabel("x")
ax.set_ylabel("u(x, 0)")
ax.set_title("Počáteční podmínka")
ax.legend()

plt.tight_layout()
plt.show()

## 3. Definice MLP modelu

Jednoduchá síť: 3 skryté vrstvy po 64 neuronech, aktivace Tanh.

```
Vstup: 2 neurony (x, t)
Linear(2 → 64) + Tanh
Linear(64 → 64) + Tanh
Linear(64 → 64) + Tanh
Linear(64 → 1) → Výstup: predikovaná teplota u(x, t)
```

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_size=64, n_layers=3):
        super().__init__()
        
        layers = []
        layers.append(nn.Linear(2, hidden_size))
        layers.append(nn.Tanh())
        
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_size, 1))
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

model = MLP(hidden_size=64, n_layers=3).to(device)
print(model)
print(f"Počet parametrů: {sum(p.numel() for p in model.parameters()):,}")

## 4. Trénování — pouze data loss

Model se učí jen z dvojic $(x, t) \to u$, bez fyzikální rovnice.

**💡 Tip:** Přidejte physics loss (PINN) pro výrazně lepší výsledky — viz skeleton níže.

In [ ]:
N_EPOCHS = 10000
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3000, gamma=0.5)
mse = nn.MSELoss()

history = []

print("Začínám trénování...")

for epoch in range(N_EPOCHS):
    optimizer.zero_grad()
    
    u_pred = model(X_all)
    loss = mse(u_pred, y_all)
    
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    history.append(loss.item())
    
    if (epoch + 1) % 1000 == 0:
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch+1:5d}/{N_EPOCHS} | Loss: {loss.item():.6f} | LR: {lr_now:.1e}")

print("\nTrénování dokončeno!")

## 5. Skeleton pro PINN loss (k doplnění)

Zde je kostra pro přidání physics loss. Doplňte výpočet residuálu rovnice vedení tepla
pomocí `torch.autograd.grad` a přidejte ho do celkové loss funkce.

In [ ]:
# === SKELETON PRO PINN LOSS (k doplnění) ===
#
# def physics_residual(model, x_col, alpha_val):
#     """
#     Spočítá residual rovnice vedení tepla:
#     r = du/dt - alpha * d²u/dx²
#     """
#     x_col = x_col.requires_grad_(True)
#     u = model(x_col)
#
#     # TODO: Spočítejte ∂u/∂t a ∂²u/∂x² pomocí torch.autograd.grad
#     # grad_u = torch.autograd.grad(u, x_col, ...)
#     # du_dx = ...
#     # du_dt = ...
#     # d2u_dx2 = ...
#
#     # residual = du_dt - alpha_val * d2u_dx2
#     # return residual
#
# Kolokační body:
# x_col = torch.rand(N_COLLOCATION, 2, device=device)
# residual = physics_residual(model, x_col, alpha)
# loss_physics = torch.mean(residual ** 2)
#
# Celková PINN loss:
# loss = loss_data + lambda_boundary * loss_boundary + lambda_initial * loss_initial + lambda_physics * loss_physics

print("Skeleton pro PINN loss — doplňte a integrujte do trénovací smyčky.")

## 6. Průběh trénování

In [ ]:
plt.figure(figsize=(10, 5))
plt.semilogy(history, alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.title("Průběh trénování — data-only MLP")
plt.grid(True, alpha=0.3)
plt.show()

## 7. Vizualizace predikce

In [ ]:
n_grid = 100
x_grid = np.linspace(0, 1, n_grid)
t_grid = np.linspace(0, 1, n_grid)
X_mesh, T_mesh = np.meshgrid(x_grid, t_grid)

grid_points = torch.tensor(
    np.column_stack([X_mesh.ravel(), T_mesh.ravel()]),
    dtype=torch.float32, device=device
)

model.eval()
with torch.no_grad():
    u_grid = model(grid_points).cpu().numpy().reshape(n_grid, n_grid)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
c = ax.pcolormesh(T_mesh, X_mesh, u_grid, shading="auto", cmap="inferno")
plt.colorbar(c, ax=ax, label="u(x,t)")
ax.set_xlabel("t")
ax.set_ylabel("x")
ax.set_title("Predikce modelu — u(x,t)")

ax = axes[1]
for t_val in [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]:
    x_line = torch.tensor(
        np.column_stack([x_grid, np.full_like(x_grid, t_val)]),
        dtype=torch.float32, device=device
    )
    with torch.no_grad():
        u_line = model(x_line).cpu().numpy()
    ax.plot(x_grid, u_line, label=f"t={t_val:.2f}")

ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.set_title("Teplotní profily")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Generování submission.csv

In [ ]:
model.eval()
with torch.no_grad():
    u_pred_test = model(X_test).cpu().numpy().flatten()

submission = pd.DataFrame({
    "id": test_df["id"].values,
    "u": np.round(u_pred_test, 6)
})

submission.to_csv("submission.csv", index=False)
print(f"Uloženo {len(submission)} predikcí do submission.csv")
print(submission.head(10))

In [ ]:
print(f"Predikce — min: {u_pred_test.min():.4f}, max: {u_pred_test.max():.4f}, mean: {u_pred_test.mean():.4f}")
print(f"Train data — min: {all_data['u'].min():.4f}, max: {all_data['u'].max():.4f}, mean: {all_data['u'].mean():.4f}")